# EDA mô tả dữ liệu laptop P3

Phân tích chỉ đọc bộ dữ liệu P2 cố định. Nhãn suy từ số sao không phải nhãn cảm xúc do con người gán. Không chia train/test, khớp mô hình, tạo nhãn ABSA hoặc gold. Chủ dự án chấp nhận ngoại lệ giáo dục cho SemEval; nguồn/giấy phép **NOT_VERIFIED**. Giữ nguyên phân vùng SemEval gốc.

Phương pháp và diễn giải: `reports/p3_data_analysis_20261009.md`. Chạy mọi ô từ kernel mới; tìm thư mục gốc dự án từ thư mục hiện hành.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'PLAN_AI.md').is_file())
if str(root) not in sys.path: sys.path.insert(0, str(root))
from src.data.eda import build_tables, LABELS
from src.data.acquisition import sha256_file
out = root / 'reports/p3/20261009_verified'
summary = json.loads((out/'summary.json').read_text(encoding='utf-8'))
source = root/'data/processed/laptop_reviews.parquet'
assert sha256_file(source) == summary['source_sha256']
dataset = pd.read_parquet(source)
computed, tables = build_tables(dataset)
print({k: computed[k] for k in ['rows','products','metadata_brand_categories','time_min','time_max','median_words','p90_words']})

## Tổng quan, nhãn đại diện và chiều dài

Giới hạn của việc đặt trước 100 reviews/product và diagnostic/QA định hình các bản phân phối này. Số lượng ký tự và khoảng trắng từ khác với từ vựng biểu thức chính quy token; không có BERT tách token. Số lượng thương hiệu là loại siêu dữ liệu, chưa được nhà sản xuất xác minh. Nhóm gần như trùng lặp rỗng là nhóm chưa được chỉ định, không thiếu sản phẩm siêu dữ liệu.

In [ ]:
display(tables['rating'])
display(tables['sentiment'][['sentiment','reviews','review_share','median_words','helpful_mean']])
display(Image(filename=str(out/'figures/length.png')))
display(tables['missing'])

## Bằng chứng từ vựng

Tần số tài liệu đếm mỗi term/ngram một lần cho mỗi review; những từ lặp đi lặp lại không làm tăng số lượng. Bigrams/trigrams vẫn tiếp giáp trong biểu thức chính quy ban đầu token. Mật khẩu mô tả không bao giờ thay đổi văn bản đã lưu và giữ lại sự phủ định. Hỗ trợ >=30 là phương pháp phỏng đoán hiển thị EDA, không phải là ngưỡng chấp nhận mới. Mức độ phổ biến của lớp trừ-nghỉ ngơi mang tính thăm dò, không phải kiểm thử. Không sử dụng lại toàn bộ tần số kho văn bản này cho lựa chọn P4/P5 đặc trưng.

In [ ]:
lexical = tables['lexical_by_class']
for label in LABELS:
    candidates = lexical[(lexical.sentiment==label)&(lexical.n==2)&(lexical.documents>=30)]
    print(label)
    display(candidates.sort_values(['prevalence_difference','documents','term'],ascending=[False,False,True]).head(10))
display(Image(filename=str(out/'figures/distinctive_bigrams.png')))

## Sản phẩm, thương hiệu, thời gian và siêu dữ liệu

Tỷ lệ dành cho các nhóm nhỏ vẫn được giữ nguyên trong bảng nhưng không được xếp hạng. Thương hiệu >=100/product >=30/time >=100 chỉ là các phương pháp phỏng đoán có khả năng diễn giải. Khoảng Wilson giả định các quan sát nhị thức độc lập và bỏ qua các sản phẩm, lựa chọn được phân cụm, lỗi siêu dữ liệu và đại diện nhiễu. Khối lượng review không phải là doanh số sản phẩm cũng như thị phần; điểm cuối tạm thời là một phần. Giao dịch mua hàng đã được xác minh và mức độ hữu ích phản ánh mức độ hiển thị, độ tuổi, kết hợp sản phẩm và độ dài văn bản.

In [ ]:
display(tables['brands'].sort_values('reviews',ascending=False).head(12))
display(tables['products'].sort_values(['reviews','parent_asin'],ascending=[False,True]).head(12))
display(tables['years'])
display(tables['verified'])
print('Helpfulness-vs-words Spearman (descriptive):', computed['helpful_words_spearman'])
display(Image(filename=str(out/'figures/time.png')))

## Các chủ đề và ví dụ chẩn đoán laptop

Từ điển phù hợp với chủ đề ứng cử viên kiểm thử; chúng không phải là khía cạnh trích xuất vị trí cụm từ hay phân cực có điều kiện khía cạnh. Pin đề cập đến positive review vẫn có thể bị chỉ trích. Danh sách chồng chéo; tỷ lệ phần trăm không cần tính tổng bằng 100%. Đoạn mã ngữ cảnh hỗ trợ giải thích, không phải con người gán nhãn. Các ứng cử viên nhãn-nhiễu được lựa chọn có chủ ý và không mang tính đại diện.

In [ ]:
display(tables['themes'])
display(tables['theme_contexts'].head(12))
display(pd.read_csv(out/'noise_examples.csv',keep_default_na=False).head(6))
display(Image(filename=str(out/'figures/themes.png')))

## Kiểm tra và giới hạn độ tái lập

P10 bộ gold do con người gán nhãn độc lập vẫn vắng mặt. Các phát hiện chỉ dành cho laptop; nhãn đại diện nhiễu, sự thay đổi SemEval-to-Amazon platform/style và các kết luận giới hạn gold được chú thích thủ công nhỏ. Không khái quát hóa nhiều tiện ích, xếp hạng chất lượng thương hiệu hoặc khẳng định mức độ sẵn sàng sản xuất.

In [ ]:
for name, table in tables.items():
    saved = pd.read_csv(out/f'{name}.csv',keep_default_na=False)
    pd.testing.assert_frame_equal(table.reset_index(drop=True),saved,check_dtype=False,rtol=1e-8,atol=1e-9)
for relative, digest in summary['output_sha256'].items():
    assert sha256_file(out/relative)==digest
assert sha256_file(source)==summary['source_sha256']
assert not summary['splits_created'] and not summary['models_trained'] and not summary['gold_labels_created']
print('PASS: fresh-kernel tables match saved P3 outputs; source and artifacts unchanged; no later-phase work.')